# Эксперимент 5Ф+6Ф: Полный RAG-пайплайн на HotpotQA
## Сравнительный анализ методов информационного поиска и LLM-генерации

**Объединённый ноутбук**, последовательно выполняющий:

### Часть I — Задание 5Ф (методы поиска + IR-метрики)
1. Установка и импорты
2. Определение 4 retriever-классов (BM25, Dense, Hybrid, Hybrid+Reranking)
3. Загрузка HotpotQA и подготовка корпуса
4. Индексация (создание поисковых структур)
5. Прогон 100 запросов через 4 метода
6. Подсчёт IR-метрик (Precision@k, Recall@k, MRR, nDCG@k)
7. Таблицы 1-2 и графики 1-2

### Часть II — Задание 6Ф (LLM-генерация + QA-метрики)
8. Подключение к Hugging Face Inference API
9. RAG-генерация ответов через LLM для всех 4 методов
10. QA-метрики (Exact Match, F1, BERTScore)
11. LLM-judge метрики через RAGAS (Faithfulness, Answer Relevancy)
12. График 3 — сводное сравнение
13. Итоговый анализ

---

**Время выполнения:** ~2-3 часа на CPU Colab.

**Что нужно для запуска:**
- Аккаунт на [huggingface.co](https://huggingface.co) (бесплатно)
- API-токен HuggingFace (Settings → Access Tokens → New token типа Read)

**Что получится на выходе:**
- 4 таблицы (IR-метрики, время, QA-метрики, RAGAS-метрики)
- 3 графика (nDCG@10, Recall@k, сводное сравнение)
- Объединённая таблица всех метрик `all_metrics_combined.csv`

## Блок 1. Установка библиотек

Устанавливаем ВСЕ библиотеки сразу — и для части I, и для части II.

In [ ]:
# === Часть I (5Ф): retrieval + IR-метрики ===
!pip install -q sentence-transformers faiss-cpu rank-bm25 datasets numpy pandas matplotlib

# === Часть II (6Ф): LLM-генерация + QA-метрики + RAGAS ===
!pip install -q huggingface_hub          # клиент Hugging Face API
!pip install -q bert-score               # BERTScore для QA
!pip install -q ragas                    # LLM-judge метрики
!pip install -q langchain langchain-community langchain-huggingface

print('Все библиотеки установлены ✓')

## Блок 2. Импорты

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Tuple, Dict, Set

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print('Импорты OK')

## Блок 3. Определение классов retriever-ов

Здесь определяются 4 класса для разных методов поиска:
- **BM25Retriever** — лексический поиск (точные слова)
- **DenseRetriever** — семантический через SBERT-эмбеддинги
- **HybridRetriever** — объединение BM25+Dense через RRF
- **HybridRerankerRetriever** — Hybrid + Cross-Encoder реранкинг

In [ ]:
# КЛАСС 1: BM25 — лексический поиск
class BM25Retriever:
    name = 'BM25'
    
    def __init__(self):
        self.documents, self.doc_ids, self.bm25 = [], [], None
    
    @staticmethod
    def _tokenize(text):
        # Простая токенизация: слова в нижнем регистре
        return re.findall(r'\w+', text.lower())
    
    def index(self, documents):
        self.doc_ids = list(documents.keys())
        self.documents = [documents[did] for did in self.doc_ids]
        tokenized = [self._tokenize(doc) for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized)
    
    def search(self, query, k=5):
        query_tokens = self._tokenize(query)
        scores = self.bm25.get_scores(query_tokens)
        top_k_idx = np.argsort(scores)[-k:][::-1]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k_idx]

print('BM25Retriever определён')

In [ ]:
# КЛАСС 2: Dense — семантический поиск через SBERT + FAISS
class DenseRetriever:
    name = 'Dense (SBERT)'
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f'  Загрузка {model_name}...', end=' ', flush=True)
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()  # 384
        # IndexFlatIP = полный перебор с inner product
        # С нормализованными векторами IP = cosine similarity
        self.index_obj = faiss.IndexFlatIP(self.dim)
        self.doc_ids, self.documents = [], []
        print('OK')
    
    def _embed(self, texts):
        # normalize_embeddings=True делает длину вектора = 1
        return self.model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=False).astype('float32')
    
    def index(self, documents):
        self.doc_ids = list(documents.keys())
        self.documents = [documents[did] for did in self.doc_ids]
        embeddings = self._embed(self.documents)
        self.index_obj.add(embeddings)
    
    def search(self, query, k=5):
        query_emb = self._embed([query])
        scores, indices = self.index_obj.search(query_emb, k)
        return [(self.doc_ids[idx], float(scores[0][i]))
                for i, idx in enumerate(indices[0])]

print('DenseRetriever определён')

In [ ]:
# КЛАСС 3: Hybrid — объединение BM25 + Dense через RRF
class HybridRetriever:
    name = 'Hybrid (RRF)'
    
    def __init__(self, bm25, dense, rrf_k=60):
        self.bm25, self.dense, self.rrf_k = bm25, dense, rrf_k
    
    def search(self, query, k=5):
        # Берём top-20 от каждого метода
        bm25_results = self.bm25.search(query, k=20)
        dense_results = self.dense.search(query, k=20)
        
        # RRF: 1/(k + rank)
        rrf_scores = defaultdict(float)
        for rank, (doc_id, _) in enumerate(bm25_results):
            rrf_scores[doc_id] += 1.0 / (self.rrf_k + rank + 1)
        for rank, (doc_id, _) in enumerate(dense_results):
            rrf_scores[doc_id] += 1.0 / (self.rrf_k + rank + 1)
        
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: -x[1])
        return sorted_docs[:k]

print('HybridRetriever определён')

In [ ]:
# КЛАСС 4: Hybrid + Reranking — переранжирование через Cross-Encoder
class HybridRerankerRetriever:
    name = 'Hybrid + Reranking'
    
    def __init__(self, hybrid, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        self.hybrid = hybrid
        print(f'  Загрузка Cross-Encoder...', end=' ', flush=True)
        self.cross_encoder = CrossEncoder(model_name)
        self.documents_dict = dict(zip(hybrid.bm25.doc_ids, hybrid.bm25.documents))
        print('OK')
    
    def search(self, query, k=5, top_n=20):
        # Этап 1: получаем top_n кандидатов от Hybrid
        candidates = self.hybrid.search(query, k=top_n)
        candidate_ids = [doc_id for doc_id, _ in candidates]
        
        # Этап 2: реранкинг через Cross-Encoder
        pairs = [(query, self.documents_dict[did]) for did in candidate_ids]
        ce_scores = self.cross_encoder.predict(pairs, show_progress_bar=False)
        
        # Этап 3: сортируем по новым скорам
        reranked = sorted(zip(candidate_ids, ce_scores), key=lambda x: -x[1])
        return [(did, float(s)) for did, s in reranked[:k]]

print('HybridRerankerRetriever определён')

## Блок 4. Загрузка HotpotQA

HotpotQA — датасет multi-hop вопросов, требующих рассуждения по нескольким документам.
Берём dev split (~7400 вопросов) и используем первые 100 для эксперимента.

Загрузка может занять 1-2 минуты при первом запуске.

In [ ]:
from datasets import load_dataset

N_QUERIES = 100  # количество запросов для эксперимента

print(f'Загрузка HotpotQA...', end=' ', flush=True)
hotpot = load_dataset('hotpot_qa', 'distractor', split='validation',
                       trust_remote_code=True)
print(f'OK ({len(hotpot)} вопросов всего)')

hotpot_subset = hotpot.select(range(N_QUERIES))
print(f'Используем {N_QUERIES} вопросов из {len(hotpot)}')

# Посмотрим на первый вопрос
print(f'\nПример вопроса:')
print(f'  Вопрос: {hotpot_subset[0]["question"]}')
print(f'  Ответ: {hotpot_subset[0]["answer"]}')
print(f'  Документов в контексте: {len(hotpot_subset[0]["context"]["title"])}')
print(f'  Релевантных (supporting): {len(hotpot_subset[0]["supporting_facts"]["title"])}')

## Блок 5. Подготовка корпуса

Из HotpotQA нужно собрать:
- **corpus** — все документы (по 10 на запрос = ~1000 документов)
- **queries_list** — список вопросов
- **qrels** — какие документы релевантны какому запросу (по supporting_facts)

In [ ]:
corpus = {}        # {doc_id: текст}
queries_list = []  # [(query_id, текст вопроса)]
qrels = {}         # {query_id: {doc_id: 1}}
doc_counter = 0

for q_idx in range(len(hotpot_subset)):
    q = hotpot_subset[q_idx]
    query_id = f'q{q_idx}'
    queries_list.append((query_id, q['question']))
    qrels[query_id] = {}
    
    titles = q['context']['title']
    sentences_list = q['context']['sentences']
    supporting_titles = set(q['supporting_facts']['title'])
    
    for title, sentences in zip(titles, sentences_list):
        doc_text = title + '. ' + ' '.join(sentences)
        doc_id = f'doc_{doc_counter}'
        corpus[doc_id] = doc_text
        
        if title in supporting_titles:
            qrels[query_id][doc_id] = 1
        
        doc_counter += 1

print(f'Корпус готов:')
print(f'  Документов: {len(corpus)}')
print(f'  Запросов: {len(queries_list)}')
print(f'  Среднее релевантных на запрос: {np.mean([len(r) for r in qrels.values()]):.2f}')

## Блок 6. Индексация

Создаём и индексируем все 4 retriever-а.
- BM25 быстрый: ~10 секунд
- Dense: ~1-2 минуты (нужно посчитать ~1000 эмбеддингов)
- Cross-Encoder загружается дополнительно: ~30 секунд

In [ ]:
print('Индексация BM25...', end=' ', flush=True)
t0 = time.time()
bm25 = BM25Retriever()
bm25.index(corpus)
t_bm25 = time.time() - t0
print(f'OK ({t_bm25:.1f}s)')

print('Индексация Dense (SBERT)...')
t0 = time.time()
dense = DenseRetriever()
dense.index(corpus)
t_dense = time.time() - t0
print(f'  Завершено за {t_dense:.1f}s')

print('Создание Hybrid и Reranker...')
hybrid = HybridRetriever(bm25, dense)
reranker = HybridRerankerRetriever(hybrid)

retrievers = [bm25, dense, hybrid, reranker]
print(f'\nГотовы 4 retriever-а:')
for r in retrievers:
    print(f'  - {r.name}')

## Блок 7. Прогон эксперимента

Прогоняем все 100 запросов через все 4 метода. Сохраняем top-10 результат.
Это **самая долгая часть** — 20-30 минут из-за Cross-Encoder.

In [ ]:
results = {r.name: {} for r in retrievers}
search_times = {r.name: [] for r in retrievers}

K = 10  # берём top-10 для разных метрик

print(f'Прогон {len(queries_list)} запросов × {len(retrievers)} методов...')
print('Это самая долгая часть — обычно 20-30 минут.\n')

for q_idx, (query_id, query_text) in enumerate(queries_list):
    if (q_idx + 1) % 10 == 0 or q_idx == 0:
        print(f'  Обработано {q_idx + 1}/{len(queries_list)} запросов')
    
    for retriever in retrievers:
        t0 = time.time()
        search_results = retriever.search(query_text, k=K)
        search_times[retriever.name].append(time.time() - t0)
        results[retriever.name][query_id] = [did for did, _ in search_results]

print(f'\nЭксперимент завершен!')

## Блок 8. Подсчёт IR-метрик

Считаем 4 метрики:
- **Precision@k** — доля релевантных в первых k результатах
- **Recall@k** — доля найденных релевантных из всех существующих
- **MRR** — обратная позиция первого релевантного документа
- **nDCG@k** — нормализованный кумулятивный выигрыш

In [ ]:
def precision_at_k(retrieved, relevant, k):
    if k == 0: return 0.0
    hits = sum(1 for doc in retrieved[:k] if doc in relevant)
    return hits / k

def recall_at_k(retrieved, relevant, k):
    if len(relevant) == 0: return 0.0
    hits = sum(1 for doc in retrieved[:k] if doc in relevant)
    return hits / len(relevant)

def mrr_score(retrieved, relevant):
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    dcg = sum((1.0 if doc in relevant else 0.0) / np.log2(i + 1)
              for i, doc in enumerate(retrieved[:k], 1))
    n_rel = min(k, len(relevant))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, n_rel + 1))
    return dcg / idcg if idcg > 0 else 0.0

# Считаем метрики для каждого метода
metrics_data = {}

for method_name, method_results in results.items():
    p5, p10, r5, r10, mrrs, n5, n10 = [], [], [], [], [], [], []
    
    for query_id, retrieved in method_results.items():
        relevant = set(qrels[query_id].keys())
        p5.append(precision_at_k(retrieved, relevant, 5))
        p10.append(precision_at_k(retrieved, relevant, 10))
        r5.append(recall_at_k(retrieved, relevant, 5))
        r10.append(recall_at_k(retrieved, relevant, 10))
        mrrs.append(mrr_score(retrieved, relevant))
        n5.append(ndcg_at_k(retrieved, relevant, 5))
        n10.append(ndcg_at_k(retrieved, relevant, 10))
    
    metrics_data[method_name] = {
        'Precision@5': np.mean(p5),
        'Precision@10': np.mean(p10),
        'Recall@5': np.mean(r5),
        'Recall@10': np.mean(r10),
        'MRR': np.mean(mrrs),
        'nDCG@5': np.mean(n5),
        'nDCG@10': np.mean(n10),
    }

df_metrics = pd.DataFrame(metrics_data).T.round(4)
print('ТАБЛИЦА 1: IR-метрики для каждого метода')
print('=' * 80)
df_metrics

## Блок 9. Таблица 2: Время работы методов

In [ ]:
timings_data = {}
for method_name in df_metrics.index:
    times = search_times[method_name]
    timings_data[method_name] = {
        'Среднее время поиска (мс)': np.mean(times) * 1000,
        'Медианное время (мс)': np.median(times) * 1000,
        'Максимум (мс)': np.max(times) * 1000,
    }
df_times = pd.DataFrame(timings_data).T.round(1)
print('ТАБЛИЦА 2: Время выполнения поиска (мс на запрос)')
print('=' * 80)
df_times

## Блок 10. График 1: Сравнение методов по nDCG@10

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

methods = list(df_metrics.index)
ndcg_values = df_metrics['nDCG@10'].values
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

bars = ax.bar(methods, ndcg_values, color=colors,
              edgecolor='black', linewidth=1.5, alpha=0.85)
ax.set_ylabel('nDCG@10', fontsize=13, fontweight='bold')
ax.set_xlabel('Метод поиска', fontsize=13, fontweight='bold')
ax.set_title('Сравнение методов информационного поиска по nDCG@10\n'
             '(HotpotQA, подвыборка 100 запросов)', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(ndcg_values) * 1.20)
ax.grid(axis='y', alpha=0.3, linestyle='--')

for bar, value in zip(bars, ndcg_values):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.01,
            f'{value:.4f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('plot1_ndcg.png', dpi=150, bbox_inches='tight')
plt.show()
print('График 1 сохранён: plot1_ndcg.png')

## Блок 11. График 2: Зависимость Recall@k от k

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

k_values = [1, 3, 5, 10]
markers = ['o', 's', '^', 'D']
linestyles = ['-', '--', '-.', ':']

for i, method_name in enumerate(methods):
    method_results = results[method_name]
    recalls = []
    for k in k_values:
        avg = np.mean([recall_at_k(method_results[qid], set(qrels[qid].keys()), k)
                       for qid in method_results.keys()])
        recalls.append(avg)
    
    ax.plot(k_values, recalls, marker=markers[i], linestyle=linestyles[i],
            linewidth=2.5, markersize=10, label=method_name, color=colors[i])

ax.set_xlabel('k — количество возвращаемых документов',
              fontsize=13, fontweight='bold')
ax.set_ylabel('Recall@k (среднее по 100 запросам)',
              fontsize=13, fontweight='bold')
ax.set_title('Зависимость Recall@k от k для разных методов поиска\n'
             '(HotpotQA, подвыборка 100 запросов)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xticks(k_values)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('plot2_recall_k.png', dpi=150, bbox_inches='tight')
plt.show()
print('График 2 сохранён: plot2_recall_k.png')

## Блок 12. Сохранение результатов

Сохраняем таблицы и графики. Скачайте их из Colab (Files → Download).

In [ ]:
# Сохраняем таблицы в CSV
df_metrics.to_csv('table1_metrics.csv', encoding='utf-8')
df_times.to_csv('table2_times.csv', encoding='utf-8')

print('Сохранены файлы:')
print('  - table1_metrics.csv (Таблица 1: IR-метрики)')
print('  - table2_times.csv (Таблица 2: время работы)')
print('  - plot1_ndcg.png (График 1)')
print('  - plot2_recall_k.png (График 2)')
print('\nСкачайте их через панель Files (слева) — там значки папки')
print('Нажмите на файл правой кнопкой → Download')

## Блок 13. Краткий анализ

Автоматическая печать выводов на основе полученных метрик.
Эти цифры пригодятся для отчёта в Word.

In [ ]:
print('=' * 70)
print('АНАЛИЗ РЕЗУЛЬТАТОВ ДЛЯ ОТЧЁТА')
print('=' * 70)

print('\n1. Лучший метод по каждой метрике:')
for metric in df_metrics.columns:
    best = df_metrics[metric].idxmax()
    val = df_metrics[metric].max()
    print(f'   {metric}: {best} ({val:.4f})')

print('\n2. Подтверждение гипотезы (Hybrid+Reranking лучший):')
best_ndcg = df_metrics['nDCG@10'].idxmax()
if best_ndcg == 'Hybrid + Reranking':
    print('   ✓ ПОДТВЕРЖДЕНО: Hybrid+Reranking лидер по nDCG@10')
else:
    print(f'   ⚠ Лидер: {best_ndcg} (не Hybrid+Reranking)')

print('\n3. Прирост Hybrid+Reranking над BM25:')
bm25_ndcg = df_metrics.loc['BM25', 'nDCG@10']
rerank_ndcg = df_metrics.loc['Hybrid + Reranking', 'nDCG@10']
if bm25_ndcg > 0:
    gain = (rerank_ndcg - bm25_ndcg) / bm25_ndcg * 100
    print(f'   nDCG@10: {bm25_ndcg:.4f} → {rerank_ndcg:.4f} (+{gain:.1f}%)')

print('\n4. Цена реранкинга по времени:')
rerank_time = df_times.loc['Hybrid + Reranking', 'Среднее время поиска (мс)']
hybrid_time = df_times.loc['Hybrid (RRF)', 'Среднее время поиска (мс)']
print(f'   Hybrid: {hybrid_time:.1f} мс/запрос')
print(f'   Hybrid+Reranking: {rerank_time:.1f} мс/запрос')
if hybrid_time > 0:
    slowdown = rerank_time / hybrid_time
    print(f'   Замедление: в {slowdown:.1f} раз')

print('\n' + '=' * 70)
print('ИСПОЛЬЗУЙТЕ ЭТИ ЦИФРЫ ДЛЯ ОТЧЁТА В WORD')
print('=' * 70)

---

# ЧАСТЬ II — Задание 6Ф (LLM-генерация и QA-оценка)

Теперь, когда у нас есть результаты IR-поиска по 4 методам, расширяем эксперимент:
- Подключаем LLM (через Hugging Face Inference API) для генерации ответов
- Считаем QA-метрики (EM, F1, BERTScore)
- Считаем LLM-judge метрики (Faithfulness, Answer Relevancy через RAGAS)

## Блок 14. Подключение к Hugging Face Inference API

**Hugging Face** — крупнейший хаб open-source моделей машинного обучения. Через Inference API можно бесплатно использовать готовые модели для генерации текста.

**Используемая модель:** `meta-llama/Llama-3.2-3B-Instruct` — небольшая, но качественная модель от Meta, оптимизированная для следования инструкциям.

**Альтернативы:**
- `mistralai/Mistral-7B-Instruct-v0.3` — Mistral 7B
- `Qwen/Qwen2.5-7B-Instruct` — Qwen 2.5 от Alibaba
- `google/gemma-2-2b-it` — Gemma от Google

**Как получить токен:**
1. Зарегистрируйтесь на [huggingface.co](https://huggingface.co)
2. Settings → Access Tokens → New token
3. Тип: «Read», имя любое, нажмите «Generate»
4. Скопируйте токен `hf_...` и вставьте ниже

In [ ]:
from huggingface_hub import InferenceClient

# ВСТАВЬТЕ СВОЙ ТОКЕН СЮДА:
HF_TOKEN = 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'  # ← ЗАМЕНИТЬ на ваш токен

# Выбираем модель. Можете попробовать разные:
MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
# Альтернативы:
# MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
# MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

client = InferenceClient(model=MODEL_NAME, token=HF_TOKEN)

# Проверяем подключение
test_response = client.chat_completion(
    messages=[
        {'role': 'user', 'content': 'Say hello in one short sentence.'}
    ],
    max_tokens=50
)

print('Ответ LLM:')
print(test_response.choices[0].message.content)
print('\n✓ Hugging Face Inference API подключен')

## Блок 15. Функция RAG-генерации

Реализуем главную функцию полного RAG-пайплайна:
1. Получаем top-k документов от retriever-а
2. Формируем контекст из этих документов
3. Передаём контекст + вопрос в LLM через HF API
4. Получаем ответ

In [ ]:
RAG_PROMPT_TEMPLATE = '''Answer the question using only the information from the context below. 
If the context does not contain the answer, respond with "Insufficient information".
Be concise. Provide only the answer without explanations.

Context:
{context}

Question: {question}

Answer:'''

def generate_rag_answer(question: str, retrieved_docs: list,
                       corpus: dict, k: int = 5) -> str:
    '''Генерирует ответ через Hugging Face Inference API.'''
    
    # 1. Собираем контекст из топ-k документов
    context_parts = []
    for i, doc_id in enumerate(retrieved_docs[:k], 1):
        doc_text = corpus[doc_id]
        if len(doc_text) > 1500:
            doc_text = doc_text[:1500] + '...'
        context_parts.append(f'[Document {i}]: {doc_text}')
    
    context = '\n\n'.join(context_parts)
    
    # 2. Формируем промпт
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    
    # 3. Вызываем HF Inference API
    try:
        response = client.chat_completion(
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=150,    # короткие фактические ответы
            temperature=0.1,   # низкая температура для стабильности
        )
        answer = response.choices[0].message.content.strip()
    except Exception as e:
        print(f'  Ошибка HF API: {e}')
        answer = ''
    
    return answer

print('Функция generate_rag_answer определена ✓')

# Тестируем на одном вопросе
test_q = queries_list[0][1]
test_method = list(results.keys())[3]  # Hybrid + Reranking
test_query_id = queries_list[0][0]
test_docs = results[test_method][test_query_id]

test_answer = generate_rag_answer(test_q, test_docs, corpus, k=5)
print(f'\nТестовый вопрос: {test_q}')
print(f'Ответ модели: {test_answer}')
print(f'Эталонный ответ: {hotpot_subset[0]["answer"]}')

## Блок 16. Прогон RAG-генерации для всех методов

Для каждого из 4 методов поиска генерируем ответы через LLM для всех 100 вопросов.

**Объём:** 4 метода × 100 вопросов = 400 запросов к HF API. С учётом лимита 300/час нужно делать паузы. В коде ниже стоит `time.sleep(0.5)` — это даёт ~120 запросов в минуту, что в пределах лимита.

**Предположение:** результаты из 5Ф уже в памяти (results, queries_list, corpus, hotpot_subset).

In [ ]:
# Загружаем эталонные ответы из HotpotQA
ground_truth = {}
for q_idx in range(len(hotpot_subset)):
    q = hotpot_subset[q_idx]
    query_id = f'q{q_idx}'
    ground_truth[query_id] = q['answer']

print(f'Загружено эталонных ответов: {len(ground_truth)}')
print(f'Пример: {list(ground_truth.values())[0]}')

In [ ]:
import time
from tqdm import tqdm

# Словарь для хранения ответов
generated_answers = {method: {} for method in results.keys()}

for method_name, method_results in results.items():
    print(f'\n=== Генерация для {method_name} ===')
    
    for query_id, retrieved_docs in tqdm(method_results.items()):
        question_text = dict(queries_list)[query_id]
        
        # Пауза 0.5 сек между запросами (rate limit HF)
        time.sleep(0.5)
        
        answer = generate_rag_answer(
            question=question_text,
            retrieved_docs=retrieved_docs,
            corpus=corpus,
            k=5
        )
        generated_answers[method_name][query_id] = answer

print(f'\n✓ Генерация завершена для всех методов')

# Сохраняем ответы (на случай сбоя)
import json
with open('generated_answers.json', 'w', encoding='utf-8') as f:
    json.dump(generated_answers, f, ensure_ascii=False, indent=2)
print('Ответы сохранены в generated_answers.json')

## Блок 17. QA-метрики

Реализуем три классические QA-метрики:
- **Exact Match (EM)** — точное совпадение после нормализации
- **F1 (token-level)** — гармоническое среднее точности и полноты по токенам
- **BERTScore** — семантическое сходство через BERT-эмбеддинги

In [ ]:
import re
import string
from collections import Counter

def normalize_answer(s: str) -> str:
    '''Нормализация: lowercase, удаление артиклей и пунктуации.'''
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match(prediction: str, ground_truth: str) -> float:
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score_qa(prediction: str, ground_truth: str) -> float:
    '''Token-level F1.'''
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

print('Функции EM и F1 определены ✓')

In [ ]:
from bert_score import score as bert_score_fn
import numpy as np
import pandas as pd

qa_metrics = {}

for method_name, method_answers in generated_answers.items():
    print(f'\n{method_name}:')
    em_scores, f1_scores = [], []
    predictions, references = [], []
    
    for query_id, pred_answer in method_answers.items():
        gt_answer = ground_truth[query_id]
        em_scores.append(exact_match(pred_answer, gt_answer))
        f1_scores.append(f1_score_qa(pred_answer, gt_answer))
        predictions.append(pred_answer if pred_answer else 'no answer')
        references.append(gt_answer)
    
    # BERTScore (батчевое вычисление)
    print('  Считаем BERTScore...')
    P, R, F1 = bert_score_fn(predictions, references, lang='en', verbose=False)
    bert_f1 = F1.mean().item()
    
    qa_metrics[method_name] = {
        'Exact Match': np.mean(em_scores),
        'F1 (token)': np.mean(f1_scores),
        'BERTScore F1': bert_f1
    }
    print(f'  EM={np.mean(em_scores):.4f}, F1={np.mean(f1_scores):.4f}, BERTScore={bert_f1:.4f}')

df_qa = pd.DataFrame(qa_metrics).T.round(4)
print('\n=== ТАБЛИЦА 3: QA-метрики ===')
df_qa.to_csv('table3_qa_metrics.csv', encoding='utf-8')
df_qa

## Блок 18. LLM-judge метрики через RAGAS

RAGAS оценит качество RAG-системы через LLM-судью. Считаем:
- **Faithfulness** — насколько ответ опирается на контекст
- **Answer Relevancy** — насколько ответ соответствует вопросу

В качестве LLM-судьи используем ту же модель из Hugging Face API.

In [ ]:
# Обёртка HuggingFace для совместимости с LangChain/RAGAS
from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.embeddings.base import Embeddings
import os

# Устанавливаем токен в переменную окружения
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN

# LLM-судья (используем ту же модель что и для генерации)
judge_llm = HuggingFaceEndpoint(
    repo_id=MODEL_NAME,
    task='text-generation',
    max_new_tokens=200,
    temperature=0.1,
    huggingfacehub_api_token=HF_TOKEN,
)

# Эмбеддинги (требуются для answer_relevancy)
judge_embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('Judge LLM и Embeddings готовы ✓')

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Оборачиваем для RAGAS
ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(judge_embeddings)

ragas_results = {}

# Для экономии запросов берём 30 вопросов
N_SAMPLE = 30

for method_name in generated_answers.keys():
    print(f'\n=== RAGAS для {method_name} ===')
    
    data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}
    
    method_results = results[method_name]
    method_answers = generated_answers[method_name]
    sample_query_ids = list(method_answers.keys())[:N_SAMPLE]
    
    for query_id in sample_query_ids:
        question_text = dict(queries_list)[query_id]
        retrieved_docs = method_results[query_id][:5]
        contexts = [corpus[did] for did in retrieved_docs]
        
        data['question'].append(question_text)
        data['answer'].append(method_answers[query_id] or 'no answer')
        data['contexts'].append(contexts)
        data['ground_truth'].append(ground_truth[query_id])
    
    dataset = Dataset.from_dict(data)
    
    # Запускаем оценку
    eval_result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_emb,
        show_progress=True
    )
    
    # eval_result — словарь со средними значениями
    ragas_results[method_name] = {
        'Faithfulness': float(eval_result['faithfulness']),
        'Answer Relevancy': float(eval_result['answer_relevancy'])
    }
    print(f'  Faithfulness: {ragas_results[method_name]["Faithfulness"]:.4f}')
    print(f'  Answer Relevancy: {ragas_results[method_name]["Answer Relevancy"]:.4f}')

df_ragas = pd.DataFrame(ragas_results).T.round(4)
print('\n=== ТАБЛИЦА 4: RAGAS-метрики ===')
df_ragas.to_csv('table4_ragas_metrics.csv', encoding='utf-8')
df_ragas

## Блок 19. График 3: сводное сравнение

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 7))

methods = list(df_qa.index)
metrics_to_show = ['F1 (token)', 'BERTScore F1']
metrics_ragas = ['Faithfulness', 'Answer Relevancy']

x = np.arange(len(methods))
width = 0.18

for i, metric in enumerate(metrics_to_show):
    values = df_qa[metric].values
    ax.bar(x + i*width - 1.5*width, values, width, label=metric, alpha=0.8)

for i, metric in enumerate(metrics_ragas):
    values = df_ragas[metric].values
    ax.bar(x + (i+2)*width - 1.5*width, values, width, label=metric, alpha=0.8)

ax.set_xlabel('Метод поиска', fontsize=13, fontweight='bold')
ax.set_ylabel('Значение метрики', fontsize=13, fontweight='bold')
ax.set_title('Сводное сравнение методов поиска по QA и LLM-judge метрикам\n'
             f'(HotpotQA, {N_SAMPLE} запросов для RAGAS, LLM: {MODEL_NAME.split("/")[-1]})',
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha='right')
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('plot3_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('График 3 сохранён: plot3_summary.png')

## Блок 20. Итоговый анализ

In [ ]:
print('=' * 70)
print('ИТОГОВЫЙ АНАЛИЗ ПОЛНОГО RAG-ПАЙПЛАЙНА')
print('=' * 70)

# Объединяем IR + QA + RAGAS метрики
all_metrics = pd.concat([
    df_metrics[['nDCG@10', 'MRR']],
    df_qa[['F1 (token)', 'BERTScore F1']],
    df_ragas
], axis=1)

print('\nСводная таблица всех метрик:')
print(all_metrics.to_string())

# Лидер по каждой метрике
print('\nЛидер по каждой метрике:')
for col in all_metrics.columns:
    best = all_metrics[col].idxmax()
    val = all_metrics[col].max()
    print(f'  {col}: {best} ({val:.4f})')

all_metrics.to_csv('all_metrics_combined.csv', encoding='utf-8')
print('\n✓ Полные результаты сохранены в all_metrics_combined.csv')
print(f'\nИспользованная LLM: {MODEL_NAME}')